In [ ]:

from typing_extensions import TypedDict, Annotated
from typing import Dict, List
import os

from langgraph.graph.message import add_messages
import operator

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from pydantic import BaseModel, Field

from scripts import my_tools

from dotenv import load_dotenv
load_dotenv()

BASE_URL = "http://localhost:11434"

# LLM_MODEL = "qwen3"
LLM_MODEL = "gpt-oss"

MAX_ITERATIONS = 3
llm = ChatOllama(
    model=LLM_MODEL, 
    base_url=BASE_URL)

In [ ]:
class Reflection(BaseModel):
    """Critique of current answer"""
    missing: str = Field(description="What critical information is missing or incomplete")
    superfluous: str = Field(description="What information is unnessary or redundant")

class Answer(BaseModel):
    """Answer with inline citation, reflection and search queries"""
    answer: str = Field(description="Detailed answer with inline citation [1], [2] with reference list at the end")
    reflection: Reflection = Field(description="Critical reflection on the answer")
    search_queries: List[str] = Field(default_factory=list,description="1-3 search queries if more information needed, empty is complete")
    is_complete: bool = Field(default=False, description="True if answer is complete and no more searches needed")

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[List, operator.add]
    iteration_count: int
    retrieved_docs: str
    search_queries: List[str]
    is_complete: bool

In [ ]:
def draft_node(state:AgentState):
    llm_structured = llm.with_structured_output(Answer)
    system_prompt = """You are an expert financial document researcher.
        TASK:
        1. Provide detailed answer (~250 words) to user's question
        2. Use Markdown formatting (headings, bullets, tables, bold)
        3. Reflect critically: identify missing and superfluous information
        4. Generate 1-3 specific search queries to retrieve missing information
        Format your answer with inline citations [1], [2] if you have any prior knowledge to cite, otherwise note what information you need.
        Output JSON matching this schema:
        {
        "answer": "Your detailed answer with inline citations [1] and references at the end...",
        "reflection": {
            "missing": "What information is missing...",
            "superfluous": "What is unnecessary..."
        },
        "search_queries": ["specific query 1", "specific query 2"],
        "is_complete": false
        }""" 
    system_msg = SystemMessage(system_prompt)
    messages = [system_msg] + state['messages']  #concatenazione
    response = llm_structured.invoke(messages)
    text_response = f"""
        **Answer**: {response.answer}\n\n
        **Reflection** - Missing: {response.reflection.missing}\n\n
        **Reflection** - Superfluous: {response.reflection.superfluous}\n\n
        **Search Queries**: {','.join(response.search_queries)}"""
    ai_msg = AIMessage(text_response)
    print(f"[DRAFT] Generated answer with {len(response.search_queries)}")
    return {
        'messages': [ai_msg],
        'iteration_count': 1,
        'search_queries': response.search_queries
    }  #update stato shared

def retrieve_node(state: AgentState):
    print(f"[RETRIEVE] Fetching documents")
    search_queries = state.get('search_queries', [])
    if not search_queries:
        print(f"[RETRIEVE] No search queries are found. skipping the retrieval part.")
        return {'retrieved_docs': 'No document is retrieved as there is no search query.'}
    all_retrieved_text = []
    for idx, query in enumerate(search_queries, 1):
        print(f"[RETRIEVE] {idx} Query: {query}")
        # 3(Reranking) -> 3*10(Retrieval) -> 3*10*20 (MMR)
        result = my_tools.retrieve_docs.invoke({'query': query, 'k': 3})
        text = f"\n---- Query {idx}: {query}\n\nResult:\n{result}"
        all_retrieved_text.append(text)
    combined_result = "\n\n".join(all_retrieved_text)
    # Save for debugging
    os.makedirs('debug_logs', exist_ok=True)  #exist_ok non da errore se la cartella esiste gia
    with open('debug_logs/reflexion_agentic_rag.md', 'w', encoding='utf-8') as f:  #apre file in write-mode con encoding target per testo
        f.write(combined_result)
    return {
        'retrieved_docs': combined_result
    } #update stato shared

def revise_node(state: AgentState):
    print(f"[REVISE] Revise - Iteration {state.get('iteration_count', 1)}")
    llm_structured = llm.with_structured_output(Answer)
    system_prompt = """You are an expert financial document researcher.
        TASK:
        1. Write DETAILED answer (~250-300 words) with MARKDOWN formatting (## headings, **bold**, bullets, tables)
        2. Include inline citations [1], [2] in the answer text
        3. Add reference list at the end: "[1] Company: x, Year: y, Quarter: z, Page: n"
        4. Critically reflect on what's missing or superfluous
        5. Generate 2-3 SPECIFIC search queries if information is incomplete
        CRITICAL - Search Query Rules:
        - Use specific keywords: company name, year, quarter, metric type
        - Good: "Amazon q1 q2 2023 quarterly revenue", "Amazon 2023 AWS segment earnings"
        - Bad: "Amazon revenue", "financial data"
        DECISION LOGIC:
        Ask yourself:
        - Do I have complete quarterly breakdown? (Q1, Q2, Q3, Q4)
        - Do I have segment-wise data if relevant?
        - Do I have year-over-year comparisons?
        - Do I have all metrics requested (revenue, income, margins, etc.)?
        - Do I have all companies mentioned in the question?
        If ALL needed information present:
        {
        "answer": "## Complete answer with citations [1][2]...\n\n**References:**\n[1] Company: amazon, Year: 2023, Page: 5",
        "is_complete": true,
        "search_queries": [],
        "reflection": {"missing": "None", "superfluous": "None"}
        }
        If MISSING any required data:
        {
        "answer": "## Answer with available data [1]...\n\n**References:**\n[1] Company: amazon, Year: 2023, Page: 5",
        "is_complete": false,
        "search_queries": ["specific query 1", "specific query 2"],
        "reflection": {"missing": "List exact gaps", "superfluous": "None"}
        }
        MANDATORY: If is_complete=false, you MUST provide 2-3 specific search_queries."""
    query_prompt = f"""
        Retrieved document:
        {state.get('retrieved_docs', 'No doc found.')}
        Revise your answer using these documents. Output JSON only data.
        """
    system_msg = SystemMessage(system_prompt)
    messages = [system_msg] + state['messages'] + [HumanMessage(query_prompt)]
    response = llm_structured.invoke(messages)
    if not response.is_complete and not response.search_queries:
        print(f"[REVISE] WARNING - No search queries provided but answer is incomplete. Forcing completion.")
        response.is_complete=True
    print(f"[REVISE] Complete: {response.is_complete}")
    print(f"[REVISE] New Queries: {'\n'.join(response.search_queries)}")
    text_response = f"""
        **Answer**: {response.answer}\n\n
        **Reflection** - Missing: {response.reflection.missing}\n\n
        **Reflection** - Superfluous: {response.reflection.superfluous}\n\n
        **Search Queries**: {','.join(response.search_queries)}\n\n
        **Status** {'Complete' if response.is_complete else 'Needs more information'}"""
    ai_msg = AIMessage(text_response)
    return {
        'messages': [ai_msg],
        'iteration_count': state.get('iteration_count', 1) + 1,
        'search_queries': response.search_queries,
        'is_complete': response.is_complete
    } #update stato shared

In [ ]:
def should_continue(state: AgentState):
    iteration_count = state.get('iteration_count', 0)
    is_complete = state.get('is_complete', False)
    search_queries = state.get('search_queries', [])
    if is_complete or not search_queries or iteration_count >= MAX_ITERATIONS:
        print(f"[ROUTER] iteration count: {iteration_count}\nis_complete: {is_complete}\nsearch_queries: {search_queries}")
        return END
    print(f"[ROUTER] Iteration {iteration_count} - continue to retrieve")
    return 'retrieve'

In [ ]:
def create_reflexion_agent():
    builder = StateGraph(AgentState)
    # add nodes
    builder.add_node('draft', draft_node)
    builder.add_node('retrieve', retrieve_node)
    builder.add_node('revise', revise_node)
    # define edges
    builder.add_edge(START, 'draft')
    builder.add_edge('draft', 'retrieve')
    builder.add_edge('retrieve', 'revise')
    # router logic
    builder.add_conditional_edges('revise', should_continue, ['retrieve', END])
    return builder.compile()

agent = create_reflexion_agent()
agent

In [ ]:
query = "what is amazon's balance sheet in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result['messages'][-1].pretty_print()


query = "Show segment wise earning for Iphones and Macbooks in 2023"
result = agent.invoke({'messages': [HumanMessage(query)]})
result['messages'][-1].pretty_print()

query = "Compare the Amazon's and Apple's revenue of 2024 Q1?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result['messages'][-1].pretty_print()
